# Runbook 1.4.a: Voice/image cloning for impersonation  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 1.4.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the classifier a call-center security tool would use to flag potentially cloned or deepfake
calls. Given a dictionary of observable signals about an inbound call or video session, return
whether the interaction is `"suspect"` (likely AI-cloned) or `"ok"` (no rule triggered).

**Lesson objective.** Identify the signals that suggest a voice or image has been AI-cloned and describe why deepfakes make identity verification harder for defenders.

### What comes in
A dict with boolean-valued keys describing what was observed. Missing keys are treated as False.
Possible keys: `"unverified_channel"`, `"urgent_money_request"`, `"audio_artifact"`,
`"no_shared_secret"`, `"caller_id_spoofed"`, `"video_face_mismatch"`, `"refused_callback"`.

### What must come out
The string `"suspect"` if any rule fires, or `"ok"` if none do.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
{"no_shared_secret": True}                                             -> "suspect"
{"audio_artifact": True, "urgent_money_request": True}                -> "suspect"
{"caller_id_spoofed": True, "unverified_channel": True}               -> "suspect"
{"urgent_money_request": True}                                         -> "ok"
{}                                                                     -> "ok"
{"audio_artifact": True}                                               -> "ok"
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(signals: dict) -> str
```

#### Input (`signals`)
A `dict` with one or more of the following string keys (all optional; missing keys are treated
as absent/False):

| Key | Type | Meaning |
|-----|------|---------|
| `"unverified_channel"` | bool | Call came in on an unexpected or unregistered channel |
| `"urgent_money_request"` | bool | Caller requests money, wire transfer, or credentials urgently |
| `"audio_artifact"` | bool | Robotic cadence, missing background noise, or flat emotional baseline |
| `"no_shared_secret"` | bool | Caller could not supply the pre-agreed code word when asked |
| `"caller_id_spoofed"` | bool | CallerID does not match the authentic registered number |
| `"video_face_mismatch"` | bool | Lighting or sync artifact visible on video |
| `"refused_callback"` | bool | Caller declined when offered a callback on the official number |

#### Output (`result`)
A single string: `"suspect"` or `"ok"`.

#### Classification rules (applied in order; first match wins)
1. If `"no_shared_secret"` is True -> `"suspect"`.
2. If `"refused_callback"` is True -> `"suspect"`.
3. If `"audio_artifact"` is True AND `"urgent_money_request"` is True -> `"suspect"`.
4. If `"caller_id_spoofed"` is True AND `"unverified_channel"` is True -> `"suspect"`.
5. If `"video_face_mismatch"` is True -> `"suspect"`.
6. If `"urgent_money_request"` is True AND `"unverified_channel"` is True -> `"suspect"`.
7. Otherwise -> `"ok"`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Missing keys in `signals` are equivalent to `False`.
- Output is always exactly `"suspect"` or `"ok"`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Dict


def solve(signals: Dict[str, bool]) -> str:
    g = signals.get

    if g("no_shared_secret"):
        return "suspect"

    if g("refused_callback"):
        return "suspect"

    if g("audio_artifact") and g("urgent_money_request"):
        return "suspect"

    if g("caller_id_spoofed") and g("unverified_channel"):
        return "suspect"

    if g("video_face_mismatch"):
        return "suspect"

    if g("urgent_money_request") and g("unverified_channel"):
        return "suspect"

    return "ok"


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_no_shared_secret_suspect():
    assert impl.solve({"no_shared_secret": True}) == "suspect"


def test_refused_callback_suspect():
    assert impl.solve({"refused_callback": True}) == "suspect"


def test_audio_plus_money_suspect():
    assert impl.solve({"audio_artifact": True, "urgent_money_request": True}) == "suspect"


def test_spoofed_plus_unverified_suspect():
    assert impl.solve({"caller_id_spoofed": True, "unverified_channel": True}) == "suspect"


def test_video_face_mismatch_suspect():
    assert impl.solve({"video_face_mismatch": True}) == "suspect"


def test_money_plus_unverified_suspect():
    assert impl.solve({"urgent_money_request": True, "unverified_channel": True}) == "suspect"


def test_empty_ok():
    assert impl.solve({}) == "ok"


def test_single_money_ok():
    assert impl.solve({"urgent_money_request": True}) == "ok"


def test_audio_only_ok():
    assert impl.solve({"audio_artifact": True}) == "ok"


def test_missing_keys_treated_false():
    assert impl.solve({"caller_id_spoofed": True}) == "ok"


def test_all_false_ok():
    signals = {k: False for k in [
        "unverified_channel", "urgent_money_request", "audio_artifact",
        "no_shared_secret", "caller_id_spoofed", "video_face_mismatch", "refused_callback",
    ]}
    assert impl.solve(signals) == "ok"


def test_rule1_takes_priority():
    assert impl.solve({
        "no_shared_secret": True,
        "audio_artifact": True,
        "urgent_money_request": True,
        "refused_callback": True,
    }) == "suspect"


def test_unverified_channel_alone_ok():
    assert impl.solve({"unverified_channel": True}) == "ok"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve({"no_shared_secret": True}),
     "suspect"),
    (lambda: solve({"audio_artifact": True, "urgent_money_request": True}),
     "suspect"),
    (lambda: solve({"caller_id_spoofed": True, "unverified_channel": True}),
     "suspect"),
    (lambda: solve({"urgent_money_request": True}),
     "ok"),
    (lambda: solve({}),
     "ok"),
    (lambda: solve({"audio_artifact": True}),
     "ok"),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it